# 1. Introducción

# 2. Adquisición

In [1]:
from collections import Counter
from pathlib import Path
import csv

from collector import CORPUS, CATEGORIES, FIELDS, FEEDS, WIKINEWS, read_corpus
from retrieval import NORMALIZED, VECTORS, clean_text, load_nlp, normalize_text, process_corpus, search_news
import joblib


def table(headers, rows):
    widths = [len(str(x)) for x in headers]
    rows = [[str(x) for x in row] for row in rows]
    for row in rows:
        widths = [max(w, len(v)) for w, v in zip(widths, row)]
    print(' | '.join(str(v).ljust(w) for v, w in zip(headers, widths)))
    print('-+-'.join('-' * w for w in widths))
    for row in rows:
        print(' | '.join(v.ljust(w) for v, w in zip(row, widths)))

raw_rows = read_corpus(CORPUS)
print('Corpus:', CORPUS.relative_to(Path.cwd()))
print('Registros:', len(raw_rows))
print('Campos:', ', '.join(FIELDS))
print('Fuentes:', ', '.join(sorted({row['source'] for row in raw_rows})))
print('Categorías:', ', '.join(CATEGORIES))
table(['Fuente configurada', 'Feeds'], [[name, ', '.join(urls)] for name, urls in FEEDS.items()])
table(['Categoría', 'Archivo Wikinews'], list(WIKINEWS.items()))

Corpus: data/news_raw_corpus.tsv
Registros: 2000
Campos: id, title, description/body, source, category, date, url, retrieved_at, author, license, license_url
Fuentes: EFF, Global Voices, Wikinews
Categorías: finanzas, deportes, tecnologia
Fuente configurada | Feeds                                                                                  
-------------------+----------------------------------------------------------------------------------------
finanzas           | https://globalvoices.org/-/topics/economics-business/feed/                             
deportes           | https://globalvoices.org/-/topics/sport/feed/                                          
tecnologia         | https://globalvoices.org/-/topics/technology/feed/, https://www.eff.org/rss/updates.xml
Categoría  | Archivo Wikinews               
-----------+--------------------------------
finanzas   | Category:Economy and business  
deportes   | Category:Sports                
tecnologia | Category:Science and te

# 3. Corpus original

In [2]:
table(['ID', 'Fuente', 'Categoría', 'Fecha', 'Título'],
      [[r['id'], r['source'], r['category'], r['date'], r['title']] for r in raw_rows[:5]])
table(['Fuente', 'Registros'], sorted(Counter(r['source'] for r in raw_rows).items()))
table(['Categoría', 'Registros'], sorted(Counter(r['category'] for r in raw_rows).items()))

ID                    | Fuente        | Categoría  | Fecha      | Título                                                                                                             
----------------------+---------------+------------+------------+--------------------------------------------------------------------------------------------------------------------
Nb52ef31d6f481a85dba5 | EFF           | tecnologia | 2026-10-16 | In Memoriam: Barry Steinhardt                                                                                      
N8002e77bdcfaa28419f1 | Global Voices | tecnologia | 2026-10-15 | How authoritarian states are building an international market for digital surveillance                             
N067c55f3ab4d13725c08 | EFF           | tecnologia | 2026-10-15 | Our Lawsuit Against the Trump Administration’s Social Media Surveillance Moves Forward. Here’s What the Court Said.
N7bb52e0b51a67d884772 | EFF           | tecnologia | 2026-10-14 | DHS Inspector General Fi

# 4. Normalización

In [3]:
sample = next(r for r in raw_rows if len(r['description/body'].split()) > 20)
text = sample['description/body']
nlp = load_nlp()
doc = nlp(clean_text(text))

def kept(token):
    lemma = token.lemma_.lower()
    from retrieval import STOP_POS
    return (not token.is_space and not token.is_punct and token.tag_ != 'POS'
            and lemma not in {"'s", '’s'} and any(ch.isalnum() for ch in lemma)
            and not (token.is_stop and (token.pos_ in STOP_POS
                 or (token.tag_ == 'DT' and token.lower_ in {'a', 'an', 'the'}))))

print('Texto original:', text[:360] + ('…' if len(text) > 360 else ''))
print('Texto normalizado:', normalize_text(text))
table(['Token', 'POS', 'Lema', 'Conservar'],
      [[t.text, t.pos_, t.lemma_.lower(), 'sí' if kept(t) else 'no']
       for t in list(doc)[:24]])

Texto original: Barry Steinhardt, a lifelong privacy champion and an early leader in the battle for digital civil liberties who served as EFF’s president in 1998-99, died last week at age 73. A Connecticut native, Steinhardt led the Vermont and Pennsylvania affiliates of the American Civil Liberties Union before becoming associate director of the national ACLU. There, he he…
Texto normalizado: barry steinhardt lifelong privacy champion early leader battle digital civil liberty serve eff president 1998 99 die last week age 73 connecticut native steinhardt lead vermont pennsylvania affiliate american civil liberties union become associate director national aclu there help spearhead fight to overturn much communications decency act result u.s. supreme court landmark reno v. aclu rule 1997 internet be still young steinhardt become one first attorney to specialize intersection technology privacy early 1998 take leave absence aclu to become eff president then return aclu 1999 2002 found aclu

# 5. Corpus normalizado

In [4]:
# Se ejecuta la normalización y se reconstruyen solo las representaciones del corpus local.
spaces_summary = process_corpus()
normalized_rows = read_corpus(NORMALIZED)
print('Archivo:', NORMALIZED.relative_to(Path.cwd()))
print('Registros:', len(normalized_rows))
table(['ID', 'Título original', 'Título normalizado'],
      [[r['id'], r['title'], r['normalized_title']] for r in normalized_rows[:5]])

Archivo: data/news_normalized_corpus.tsv
Registros: 2000
ID                    | Título original                                                                                                     | Título normalizado                                                                         
----------------------+---------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------
Nb52ef31d6f481a85dba5 | In Memoriam: Barry Steinhardt                                                                                       | memoriam barry steinhardt                                                                  
N8002e77bdcfaa28419f1 | How authoritarian states are building an international market for digital surveillance                              | authoritarian state be build international market digital surveillance                     
N067c55

# 6. Representaciones VSM

In [5]:
bundle = joblib.load(VECTORS)
example_index = 0
matrix_examples = []
for field in ('title', 'body'):
    for n in (1, 2):
        for representation in ('frequency', 'binary', 'tfidf'):
            key = f'{field}_{n}_{representation}'
            item = bundle['spaces'][key]
            vector = item['matrix'].getrow(example_index)
            names = item['vectorizer'].get_feature_names_out()
            pairs = sorted(zip(vector.indices, vector.data), key=lambda pair: (-pair[1], names[pair[0]]))[:4]
            terms = ', '.join(f'{names[i]}={value:.3g}' for i, value in pairs)
            matrix_examples.append([field, 'unigramas' if n == 1 else 'bigramas',
                                    representation, f'{item["matrix"].shape[0]} × {item["matrix"].shape[1]}', terms])
table(['Campo', 'N-gramas', 'Representación', 'Dimensión', 'Ejemplo (doc. 1)'], matrix_examples)

Campo | N-gramas  | Representación | Dimensión     | Ejemplo (doc. 1)                                                              
------+-----------+----------------+---------------+-------------------------------------------------------------------------------
title | unigramas | frequency      | 2000 × 4763   | barry=1, memoriam=1, steinhardt=1                                             
title | unigramas | binary         | 2000 × 4763   | barry=1, memoriam=1, steinhardt=1                                             
title | unigramas | tfidf          | 2000 × 4763   | memoriam=0.6, steinhardt=0.6, barry=0.53                                      
title | bigramas  | frequency      | 2000 × 11371  | barry steinhardt=1, memoriam barry=1                                          
title | bigramas  | binary         | 2000 × 11371  | barry steinhardt=1, memoriam barry=1                                          
title | bigramas  | tfidf          | 2000 × 11371  | barry steinhardt=0.707,

## Recuperación

In [6]:
results = search_news('artificial intelligence regulation', field='body', ngram=1,
                     representation='tfidf', top_k=5, bundle=bundle)
table(['Puntaje', 'ID', 'Título'],
      [[f"{r['score']:.6f}", r['id'], r['title']] for r in results])

Puntaje  | ID                    | Título                                                                                     
---------+-----------------------+--------------------------------------------------------------------------------------------
0.315483 | Ne340df4e8c34d8657294 | The technological transformation of Colombia's armed conflict                              
0.306649 | Nb534fc18b10fc70a6178 | Who is talking about artificial intelligence in Ecuador?                                   
0.199742 | Nd96004b71171145d4aab | EFF Statement on California Governor's Executive Order on AI                               
0.127490 | N9ac3a80aa4a86d9d9ad6 | Blow out sales prices likely on mattresses as new U.S. fire-resistant standards take effect
0.121696 | Ndec0fa9bb58fb1c5c368 | European Council agrees to crypto asset regulations                                        
